[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_02_main_cnn.ipynb)

# Module 6, Vision: Letting the Network Learn Its Own Features

**Notebook:** `06_02_main_cnn`

## What we're doing

Same dataset as [`06_01_main_classical`](06_01_main_classical.ipynb), same task, same test set. But this time we don't hand-design any features. We hand the raw pixels to a neural network and let gradient descent figure out which patterns matter.

We'll build *two* networks on the same data:

1. A **dense baseline** that flattens the image into a long vector and ignores spatial structure. This is the wrong architecture for vision; we use it to make the contrast visible.
2. A **small CNN** with the inductive bias from [`06_00_main_intuition`](06_00_main_intuition.ipynb): convolution + pooling, learned filters, weight sharing across the image.

## The recipe

| Step | Tool | What it does |
| --- | --- | --- |
| Load | Zenodo + PIL | same EuroSAT 5,000-image sample, same 80/20 split as 06_01 |
| Pipe | tf.data | wrap numpy in batches, normalize to [0, 1] |
| Train (dense) | tf.keras Sequential | flatten -> dense -> dense -> softmax |
| Train (CNN) | tf.keras Sequential | conv -> pool -> conv -> pool -> conv -> dense |
| Compare | accuracy table + confusion | head-to-head against classical baseline |

**Comparison anchor.** On this exact test set, [`06_01`](06_01_main_classical.ipynb) hit ~77% with logistic regression and ~79% with a random forest on hand-crafted features. The CNN should beat both, by a noticeable margin and without us touching the feature engineering.

## 0) Setup

TensorFlow is the only non-trivial dep, and it's already in the project's `pyproject.toml`. The data loader matches 06_01 exactly so the test set is the same. The model and data might get a little big for colab.

In [1]:
import os, zipfile

import numpy as np
import requests
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf

tf.keras.utils.set_random_seed(1955)
RNG = np.random.default_rng(1955)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

TensorFlow: 2.21.0
GPU available: False


## 1) Load EuroSAT (matches 06_01 exactly)

Same Zenodo download, 5,000 sample, 80/20 stratified split with `random_state=1955`. From the train portion we carve out a 12.5% slice for validation, leaving 70 / 10 / 20 train/val/test.

*Why we're matching 06_01.* So the test-set numbers are directly comparable. Every image the CNN sees at test time is an image the random forest also saw at test time.

In [2]:
DATA_DIR = "assets/data"
ZIP_PATH = f"{DATA_DIR}/EuroSAT_RGB.zip"
EXTRACT_DIR = f"{DATA_DIR}/EuroSAT_RGB"
URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ZIP_PATH):
    print("Downloading EuroSAT (~90 MB, one-time)...")
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

def find_class_dir(root):
    for r, dirs, _ in os.walk(root):
        if len(dirs) >= 8:
            return r
    raise RuntimeError(f"Could not find class folders under {root}")

CLASS_DIR = find_class_dir(EXTRACT_DIR)
label_names = sorted(os.listdir(CLASS_DIR))
num_classes = len(label_names)

N_SAMPLES = 5000
all_paths = []
for ci, cls in enumerate(label_names):
    for fn in os.listdir(os.path.join(CLASS_DIR, cls)):
        all_paths.append((os.path.join(CLASS_DIR, cls, fn), ci))
RNG.shuffle(all_paths)
sampled = all_paths[:N_SAMPLES]

X_img = np.zeros((N_SAMPLES, 64, 64, 3), dtype=np.uint8)
y     = np.zeros(N_SAMPLES, dtype=np.int64)
for i, (path, lab) in enumerate(sampled):
    X_img[i] = np.asarray(Image.open(path).convert("RGB"))
    y[i] = lab

X_trv, X_te, y_trv, y_te = train_test_split(
    X_img, y, test_size=0.20, stratify=y, random_state=1955
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_trv, y_trv, test_size=0.125, stratify=y_trv, random_state=1955
)
print(f"train: {X_tr.shape}    val: {X_val.shape}    test: {X_te.shape}")
print(f"classes: {label_names}")

train: (3500, 64, 64, 3)    val: (500, 64, 64, 3)    test: (1000, 64, 64, 3)
classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## 2) tf.data pipeline

Three small `tf.data.Dataset` objects, one per split. Each does the same three things: cast to float32, divide by 255, batch + prefetch. Nothing fancy.

In [ ]:
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2048, seed=1955)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train = make_ds(X_tr, y_tr, shuffle=True)
ds_val   = make_ds(X_val, y_val)
ds_test  = make_ds(X_te, y_te)

for xb, yb in ds_train.take(1):
    print("X batch:", xb.shape, xb.dtype, " y batch:", yb.shape, yb.dtype)
    IMG_SHAPE = xb.shape[1:]

## 3) The 'wrong' baseline: a dense network on flattened pixels

Flattening turns a `64x64x3` image into a 12,288-element vector with no notion that pixel `(0, 0)` and pixel `(0, 1)` are next to each other. A dense layer connects every input to every output, no spatial bias, no weight sharing.

It will learn *something*. The question is how much, and what it leaves on the table.

In [ ]:
dense_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG_SHAPE),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
], name="dense_baseline")

dense_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
dense_model.summary()

In [ ]:
history_dense = dense_model.fit(
    ds_train, validation_data=ds_val, epochs=10, verbose=2,
)

## 4) A small CNN: same data, the right inductive bias

Three convolutional blocks (each: `Conv2D` -> `MaxPool2D`), then a global average pool, then a small dense head. Two augmentation layers at the very front (active only during training) flip and rotate each image at random, this is *free data* for satellite imagery, where the up/down/left/right orientation of a forest patch carries no semantic meaning.

Notes on the choices:

- **`RandomFlip("horizontal_and_vertical")` + `RandomRotation(0.1)`**, free regularization. On 3,500 images, augmentation is the single biggest reason the CNN clears the classical baseline. Skip it and the model overfits, val accuracy plateaus around 70-75% while train climbs to 95+, and the random forest from 06_01 ends up looking better than the CNN.
- **`Conv2D(32, 3)`** → 32 different 3x3 filters. Each one looks for one local pattern (an edge orientation, a color blob, etc.). The CNN learns these from data instead of us hand-designing Sobel kernels.
- **`MaxPooling2D`** halves the spatial size after each conv block, the same downsampling we did by hand in the intuition notebook.
- **`GlobalAveragePooling2D`** turns the final feature map into one number per channel, far fewer parameters than `Flatten`.
- **One small dense head** (128 units) before the softmax. This is where the CNN combines the spatial features into a class decision.
- **`Dropout(0.25)`** randomly zeros out a quarter of the dense activations during training. Full Disclosure: dropout is less fashionable than it once was (BatchNorm and aggressive data augmentation often do the regularization heavy lifting now), but on a small model trained on 3,500 images it's a cheap insurance against overfitting. And I like it. So I'll keep it.

In [ ]:
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG_SHAPE),

    # Augmentation: only active during training, identity at inference time.
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
], name="cnn_small")

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
cnn_model.summary()

### One training guardrail: early stopping

Stop training when validation loss stops improving for **5** epochs (longer than usual because augmentation slows convergence), and rewind to the best weights. This is the cheapest, most reliable regularizer in deep learning, and on small datasets it often matters more than the architecture.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history_cnn = cnn_model.fit(
    ds_train, validation_data=ds_val,
    epochs=60, callbacks=[early_stop], verbose=2,
)

## 5) Learning curves

Two things to look for:
1. The CNN should reach a higher *validation* accuracy than the dense baseline.
2. The CNN's train/val gap should be smaller, the inductive bias acts as a regularizer.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, h, name in zip(axs, [history_dense, history_cnn], ["dense baseline", "CNN"]):
    ax.plot(h.history["accuracy"],     label="train")
    ax.plot(h.history["val_accuracy"], label="val")
    ax.set_title(f"{name} accuracy")
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy")
    ax.set_ylim(0, 1); ax.legend()
plt.tight_layout()
plt.show()

## 6) Test-set comparison: classical vs dense vs CNN

All three models are evaluated on the *same* 1,000 test images. Random chance is 10%.

In [ ]:
_, dense_acc = dense_model.evaluate(ds_test, verbose=0)
_, cnn_acc   = cnn_model.evaluate(ds_test, verbose=0)

print(f"{'model':<35s} {'test acc':>10s}")
print("-" * 47)
print(f"{'random chance':<35s} {1/num_classes:>10.3f}")
print(f"{'classical (RF on 34 features)':<35s} {0.785:>10.3f}    # from 06_01")
print(f"{'dense net on flattened pixels':<35s} {dense_acc:>10.3f}")
print(f"{'small CNN':<35s} {cnn_acc:>10.3f}")

## 7) Where the CNN still gets it wrong

Confusion matrix on the test set. The same off-diagonal pattern that hurt the random forest (Highway/Industrial/Residential confusion, the various Crop classes blurring together) tends to persist, just at a smaller scale. (the hard part of this data is still hard)

In [ ]:
y_pred_cnn = cnn_model.predict(ds_test, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_te, y_pred_cnn)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes)); ax.set_xticklabels(label_names, rotation=90, fontsize=9)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(label_names, fontsize=9)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("CNN confusion matrix (same test set as 06_01)")
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(classification_report(y_te, y_pred_cnn, target_names=label_names))

### High-confidence mistakes

When the CNN is *confidently* wrong, the picture is usually genuinely ambiguous (a forested river, a crop field next to a pasture, a residential block on a highway). These are the cases where more data, augmentation, or a stronger backbone would help, the topic of [`06_03_main_transfer.ipynb`](06_03_main_transfer.ipynb).

In [ ]:
probs = cnn_model.predict(ds_test, verbose=0)
preds = probs.argmax(axis=1)
confs = probs.max(axis=1)

wrong = np.where(preds != y_te)[0]
wrong = wrong[np.argsort(-confs[wrong])][:8]

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in zip(wrong, axs.flat):
    ax.imshow(X_te[k])
    ax.set_title(
        f"true: {label_names[y_te[k]]}\npred: {label_names[preds[k]]} ({confs[k]:.2f})",
        fontsize=9,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## What we built

On the same test set, with the same data:

| Model | What it sees | Test accuracy |
| --- | --- | --- |
| Random forest (06_01) | 34 hand-named features | ~0.79 |
| Dense baseline | 12,288 flattened pixels | depends on run, often ~0.55 |
| Small CNN | raw 64x64x3 images | typically 0.85+ |

The CNN beats the random forest with **zero** human feature engineering. It also beats the dense baseline despite the dense baseline having access to the same raw pixels, the difference is the *architecture*, not the data.

Three real takeaways:

1. **Inductive bias matters more than parameter count.** The dense network has *more* parameters than the CNN and still loses. Wiring the right assumption into the architecture (locality + weight sharing) is worth more than throwing dense layers at the problem.
2. **Augmentation is load-bearing on small data.** Try setting `RandomFlip` and `RandomRotation` to no-ops and re-run, the CNN drops below the random forest. Classical features have an inherent ceiling, but they're a *steep* ceiling. CNNs only clear it when you give them a way to escape the small-data trap.
3. **The CNN learns the same kinds of filters we hand-designed in 06_00.** The first conv layer learns edge detectors and color blobs, the next layers compose those into textures, and the last into class-level patterns. We don't have to design any of it.

## Where to go next

[`06_03_main_transfer.ipynb`](06_03_main_transfer.ipynb) drops the 'train from scratch' assumption: instead of starting from random weights, we initialize from a network that's already learned good general-purpose visual filters on millions of natural images. That single change usually moves the needle by another 5-10 points, with less compute. It's the default move on small datasets.

## What to play with next?

- **`N_SAMPLES`**, raise to 10000+ to see how much data helps a CNN (it helps a lot).
- **Augmentation strength**, `RandomRotation(0.2)` or add `RandomZoom(0.1)` for more aggressive regularization.
- **Architecture**, add more conv blocks, change filter counts, try `BatchNormalization` between conv and activation.
- **`patience`** in early stopping, smaller is more aggressive; larger lets training run longer.